# CHAD GCF – Rapid Rangeland Screening

Python/Jupyter version of the original Google Earth Engine workflow.

This notebook downloads raster outputs directly to:

`C:\Users\gianluca.blaco\Documents\GIS Projects\GCF\06_Dimos_script`

**ANNOTATED VERSION** — analytical code preserved; execution outputs cleared for readability.


# Methodological guide to the workflow

This annotated version keeps the analytical logic of the original notebook unchanged.  
The code outputs have been cleared only to make the notebook easier to read and re-run.

## Overall objective

The workflow is a **rapid spatial screening for rangeland restoration suitability**. It does not directly model restoration success. Instead, it progressively narrows the territory using several independent sources of evidence:

1. **Persistent natural/semi-natural grassland** from Global Pasture Watch (GPW).
2. **Land degradation status** from Trends.Earth.
3. **Rainfall deficit evidence** from CHIRPS.
4. **Vegetation stress evidence** from a seasonal VCI derived from MODIS NDVI.
5. **Livestock concentration** from Global Pasture Watch livestock data.
6. **Flood hazard constraints** from GloFAS.
7. A final decision rule producing two screening classes:
   - `1 = moderate-only suitability`
   - `2 = conservative suitability`

The final raster is therefore a **multi-criteria screening product**, not a single-source land-cover map.

## Important methodological points before reading the code

- The final raster is exported at **30 m**, but this does **not** mean that every input criterion has 30 m information content. Some controls are much coarser (for example livestock, VCI, rainfall and flood hazard). The 30 m grid mainly preserves the geometry of the fine-resolution GPW core mask.
- ESA WorldCover is used here as a **diagnostic / consistency check**. It is not used directly in the final suitability decision.
- The rainfall criterion is relative: a year is considered dry when seasonal rainfall is at least **10% below the 2001-2020 mean**.
- The VCI is also relative: each pixel is compared with its own historical minimum and maximum seasonal vegetation condition over **2001-2020**.
- Livestock thresholds are **data-driven percentiles** within the target area and restoration-core context rather than fixed head-count thresholds.
- Missing GloFAS flood-depth pixels are converted to zero before applying the flood constraint. This operationally treats missing flood-depth information as “no modeled flood depth”, which should be documented as an assumption.


## Setup

Install dependencies once if needed:

```python
!pip install -U earthengine-api geemap geedim
```

On first use, Earth Engine authentication may open a browser window.

## Code block: imports and notebook purpose

### Methodological role
This block does not perform analysis. It loads the Python libraries needed to communicate with Google Earth Engine and to download the resulting rasters.

### Line-by-line
- `from pathlib import Path` — imports Python's path-handling class. It is used later to define and create the local output directory.
- `import ee` — imports the Google Earth Engine Python API. All remote raster processing is executed through this package.
- `import geemap` — imports `geemap`, used here mainly to download Earth Engine images as local GeoTIFF files.

The large comment header documents that this notebook reproduces the raster-producing logic of an earlier GEE workflow.


In [ ]:
# ================================================================
# CHAD GCF - RAPID RANGELAND SCREENING
# Python / Jupyter version
#
# Reproduces the raster-producing logic of the original GEE script
# and downloads GeoTIFFs directly to a local Windows folder.
#
# Required:
#   pip install -U earthengine-api geemap geedim
#
# First Earth Engine authentication only:
#   ee.Authenticate()
# ================================================================

from pathlib import Path
import ee
import geemap

## Code block 0: user parameters

### Methodological role
This is the control panel of the analysis. Thresholds and time windows defined here directly determine the screening results.

### Line-by-line

- `EE_PROJECT = "ag-recovery-gaza"` — identifies the Earth Engine / Google Cloud project used to initialize Earth Engine. The project name does not define the study area.
- `OUTPUT_DIR = Path(...)` — defines the local Windows directory where GeoTIFF outputs will be written.
- `OUTPUT_DIR.mkdir(parents=True, exist_ok=True)` — creates the directory if it does not already exist.

### Global Pasture Watch persistence parameters
- `START_YEAR = 2018`
- `END_YEAR = 2022`

These define a five-year window used to identify persistent natural/semi-natural grassland.

- `MIN_PERSISTENCE_YEARS = 4` — a pixel must satisfy the GPW natural/semi-natural criterion in at least 4 of the 5 years.
- `MIN_NATSEMI_PROB = 60` — the GPW probability of natural/semi-natural grassland must be at least 60%.

### VCI parameters
- `VCI_VEG_BAND = "NDVI"` — selects MODIS NDVI as the vegetation variable.
- `VCI_BASELINE_START_YEAR = 2001`
- `VCI_BASELINE_END_YEAR = 2020`

These twenty years form the historical reference period.

- `VCI_ANALYSIS_START_YEAR = 2021`
- `VCI_ANALYSIS_END_YEAR = 2025`

These define the five seasons screened for recent vegetation stress.

- `VCI_DROUGHT_THRESHOLD = 0.40` — pixels with seasonal VCI ≤ 0.40 are treated as vegetation-stress / drought-evidence pixels.
- `VCI_MIN_DROUGHT_YEARS = 2` — recurrent VCI stress means at least two stressed seasons during 2021-2025.

### Suitability thresholds
- `RAIN_DRY_THRESHOLD_PCT = -10` — seasonal rainfall anomaly ≤ -10% is considered evidence of a dry season.
- `FLOOD_MIN_DEPTH_M = 0.1` — 0.1 m is the strict flood-depth threshold used by the conservative screening.
- `MODERATE_MAX_FLOOD_DEPTH_M = 0.5` — moderate screening accepts modeled flood depths below 0.5 m.
- `MODERATE_MIN_VCI_DROUGHT_YEARS = 1` — moderate screening needs at least one VCI-stress year when VCI is one of the selected evidence criteria.
- `CONSERVATIVE_MIN_VCI_DROUGHT_YEARS = 2` — conservative screening requires at least two VCI-stress years.

### Download dictionary
`DOWNLOAD = {...}` is a set of switches.  
Each key controls whether a particular raster is downloaded later. `True` means “download”; `False` means “skip”.

**QA note:** `worldcover_relevant` is defined in the dictionary, but the original notebook does not contain a corresponding download block for `wc_relevant`.


In [ ]:
# ================================================================
# 0. USER PARAMETERS
# ================================================================

EE_PROJECT = "ag-recovery-gaza"

OUTPUT_DIR = Path(
    r"C:\Users\gianluca.blaco\Documents\GIS Projects\GCF\06_Dimos_script"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2018
END_YEAR = 2022

MIN_PERSISTENCE_YEARS = 4
MIN_NATSEMI_PROB = 60

# VCI parameters from the original GCF script
VCI_VEG_BAND = "NDVI"
VCI_BASELINE_START_YEAR = 2001
VCI_BASELINE_END_YEAR = 2020
VCI_ANALYSIS_START_YEAR = 2021
VCI_ANALYSIS_END_YEAR = 2025
VCI_DROUGHT_THRESHOLD = 0.40
VCI_MIN_DROUGHT_YEARS = 2

# Restoration suitability parameters
RAIN_DRY_THRESHOLD_PCT = -10
FLOOD_MIN_DEPTH_M = 0.1
MODERATE_MAX_FLOOD_DEPTH_M = 0.5
MODERATE_MIN_VCI_DROUGHT_YEARS = 1
CONSERVATIVE_MIN_VCI_DROUGHT_YEARS = 2

# What to download.
# Set any entry to False if you do not need that raster.
DOWNLOAD = {
    "gpw_persistence_count": True,
    "gpw_persistent_rangeland": True,
    "gpw_mean_probability": True,
    "worldcover_relevant": True,
    "gpw_worldcover_crop_conflict": False,
    "livestock_2022": True,
    "trends_earth_restoration_need": True,
    "core_restoration_candidate": True,
    "chirps_rain_anomaly_2025": True,
    "glofas_rp100_depth": True,
    "glofas_rp100_extent": True,
    "vci_2025": True,
    "vci_drought_count_2021_2025": True,
    "core_recurrent_vci_drought": True,
    "suitability_moderate_conservative": True,
    "suitability_conservative": True,
}

## Code block 1: Earth Engine initialization

### Methodological role
No spatial analysis occurs here. This block establishes the authenticated Earth Engine session.

### Line-by-line
- `try:` — first attempts to initialize Earth Engine using the project specified above.
- `ee.Initialize(project=EE_PROJECT)` — starts the Earth Engine session if valid credentials are already available.
- `except Exception:` — if initialization fails, the notebook assumes authentication may be required.
- `ee.Authenticate()` — launches the Earth Engine authentication procedure.
- `ee.Initialize(project=EE_PROJECT)` — initializes again after authentication.
- The two `print(...)` lines confirm the connection and show the local output folder.


In [ ]:
# ================================================================
# 1. EARTH ENGINE INITIALIZATION
# ================================================================

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized.")
print(f"Output folder: {OUTPUT_DIR}")

## Code block 2: Area of interest (AOI)

### Methodological role
This block loads the administrative/provincial geometry that defines the broad processing extent.

### Line-by-line
- `provinces = ee.FeatureCollection(...)` — loads the Earth Engine vector asset `Focus_Provinces_final_tcd`.
- `aoi = provinces.geometry()` — merges/uses the geometry of the whole FeatureCollection as the broad AOI.

At this stage the AOI is broader than the final three-province target area. A narrower `target_aoi` is created later.


In [ ]:
# ================================================================
# 2. AOI
# ================================================================

provinces = ee.FeatureCollection(
    "projects/impact-487314/assets/Focus_Provinces_final_tcd"
)

aoi = provinces.geometry()

## Code block 3: Global Pasture Watch — persistent natural/semi-natural rangeland

### Methodological objective
The purpose of this block is to identify areas that are **consistently classified as natural or semi-natural grassland**, rather than accepting a one-year classification that could reflect temporary noise or land-cover change.

### Input datasets
- `grassland_c` — annual GPW grassland class.
- `nat-semi-grassland_p` — annual probability that grassland is natural/semi-natural.

### Function `annual_natsemi(year)`

- `year = int(year)` — ensures the supplied year is a Python integer.
- `start = ee.Date.fromYMD(year, 1, 1)` — first day of the selected year.
- `end = start.advance(1, "year")` — first day of the following year; used as the exclusive end date.

#### Annual class
- `gpw_class.filterDate(start, end)` — keeps the GPW class image for that year.
- `.first()` — retrieves the first image in the annual selection.
- `.select("dominant_class")` — keeps the dominant-class band.

#### Annual natural/semi-natural probability
- `gpw_prob.filterDate(start, end)` — selects the same year from the probability collection.
- `.first()` — retrieves the annual probability image.
- `.select("probability")` — keeps the probability band.

#### Binary annual mask
- `cls.eq(2)` — tests whether GPW's dominant class equals class 2.
- `.And(prob.gte(MIN_NATSEMI_PROB))` — additionally requires probability ≥ 60%.
- `.rename("nat_semi")` — names the resulting Boolean band.

The output is therefore 1 where both conditions are satisfied and 0 elsewhere.

- `.set({"year": year, ...})` — stores the year and timestamp as image properties.

### Persistence calculation
- `annual_natsemi_collection = ...` — runs `annual_natsemi()` for every year from 2018 through 2022 and constructs an ImageCollection.
- `.sum()` — adds the annual 0/1 rasters. The resulting value is the number of years in which the pixel passed the rangeland condition.
- `.rename("GPW_natsemi_year_count")` — names the persistence-count band.
- `.clip(aoi)` — clips the raster to the broad AOI.

### Persistent rangeland mask
- `persistence_count.gte(MIN_PERSISTENCE_YEARS)` — requires at least 4 qualifying years.
- `.selfMask()` — masks zeros and keeps only qualifying pixels.
- `.rename("GPW_persistent_natsemi")` — names the final persistent-rangeland mask.

### Mean probability diagnostic
The final block filters the probability collection over 2018-2022, selects the probability band and calculates `.mean()`. This produces the average GPW natural/semi-natural probability and is useful for QA or interpretation, but it is not itself the final suitability score.


In [ ]:
# ================================================================
# 3. GLOBAL PASTURE WATCH
# ================================================================

gpw_class = ee.ImageCollection(
    "projects/global-pasture-watch/assets/ggc-30m/v1/grassland_c"
)

gpw_prob = ee.ImageCollection(
    "projects/global-pasture-watch/assets/ggc-30m/v1/nat-semi-grassland_p"
)


def annual_natsemi(year):
    year = int(year)

    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    cls = ee.Image(
        gpw_class
        .filterDate(start, end)
        .first()
    ).select("dominant_class")

    prob = ee.Image(
        gpw_prob
        .filterDate(start, end)
        .first()
    ).select("probability")

    natsemi = (
        cls.eq(2)
        .And(prob.gte(MIN_NATSEMI_PROB))
        .rename("nat_semi")
    )

    return natsemi.set({
        "year": year,
        "system:time_start": start.millis()
    })


annual_natsemi_collection = ee.ImageCollection.fromImages(
    [annual_natsemi(y) for y in range(START_YEAR, END_YEAR + 1)]
)

persistence_count = (
    annual_natsemi_collection
    .sum()
    .rename("GPW_natsemi_year_count")
    .clip(aoi)
)

persistent_natsemi = (
    persistence_count
    .gte(MIN_PERSISTENCE_YEARS)
    .selfMask()
    .rename("GPW_persistent_natsemi")
)

mean_natsemi_prob = (
    gpw_prob
    .filterDate(
        ee.Date.fromYMD(START_YEAR, 1, 1),
        ee.Date.fromYMD(END_YEAR + 1, 1, 1)
    )
    .select("probability")
    .mean()
    .clip(aoi)
    .rename("GPW_mean_natsemi_probability")
)

## Code block 4: ESA WorldCover 2021 — land-cover consistency check

### Methodological objective
WorldCover is used to compare the GPW rangeland mask with an independent land-cover product.

### Line-by-line
- `ee.ImageCollection("ESA/WorldCover/v200")` — loads ESA WorldCover v200.
- `.first()` — retrieves the WorldCover image.
- `.select("Map")` — keeps the categorical land-cover band.
- `.clip(aoi)` — clips to the broad AOI.

### Relevant land-cover classes
`wc_relevant` keeps:
- class `20` — shrubland,
- class `30` — grassland,
- class `40` — cropland.

This creates a diagnostic layer containing the land-cover types most relevant to the rangeland/cropland screening.

### Rangeland-compatible WorldCover mask
`wc_rangeland_compatible` is true for:
- class 20 (shrubland), or
- class 30 (grassland).

### Agreement layer
`gpw_wc_agreement` starts from the persistent GPW rangeland mask and keeps only pixels where WorldCover is shrubland or grassland.

### Conflict layer
`gpw_cropland_conflict` starts from persistent GPW rangeland and keeps pixels where WorldCover classifies the same location as cropland (40).

### Important methodological note
In the current script, these WorldCover layers are **diagnostic only**. Neither `gpw_wc_agreement` nor `gpw_cropland_conflict` is used in the final `core_restoration` or final suitability rules.


In [ ]:
# ================================================================
# 4. ESA WORLDCOVER 2021
# ================================================================

world_cover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
    .clip(aoi)
)

wc_relevant = world_cover.updateMask(
    world_cover.eq(20)
    .Or(world_cover.eq(30))
    .Or(world_cover.eq(40))
)

wc_rangeland_compatible = (
    world_cover.eq(20)
    .Or(world_cover.eq(30))
)

gpw_wc_agreement = (
    persistent_natsemi
    .updateMask(wc_rangeland_compatible)
    .rename("GPW_WC_rangeland_agreement")
)

gpw_cropland_conflict = (
    persistent_natsemi
    .updateMask(world_cover.eq(40))
    .rename("GPW_WC_cropland_conflict")
)

## Code block 5: Global Pasture Watch livestock — 2022

### Methodological objective
This block prepares a livestock-pressure / livestock-presence variable to support restoration targeting.

### Line-by-line
- `livestock_collection = ee.ImageCollection(...)` — loads the 1 km GPW livestock headcount dataset.
- `.filterDate("2022-01-01", "2023-01-01")` — isolates the 2022 layer.
- `.first()` — retrieves the 2022 image.
- `.clip(aoi)` — limits it to the AOI.

### Export bands
`selected_livestock` selects:
- cattle,
- sheep,
- goat,
- horse.

`selected_livestock.reduce(ee.Reducer.sum())` adds those four species pixel-by-pixel and creates `cattle_sheep_goat_horse_2022`.

`livestock_export = selected_livestock.addBands(total_selected_livestock)` retains the individual species plus their four-species total for export.

### Livestock variable used in suitability
`livestock_csg` separately selects only:
- cattle,
- sheep,
- goat,

and sums them into `cattle_sheep_goat_2022`.

**Important:** horses are present in the exported livestock raster but are **not included in the livestock threshold used later for suitability**.


In [ ]:
# ================================================================
# 5. GLOBAL PASTURE WATCH LIVESTOCK - 2022
# ================================================================

livestock_collection = ee.ImageCollection(
    "projects/global-pasture-watch/assets/gld-1km/v1/"
    "livestock-headcount-faostat_m"
)

livestock_2022 = ee.Image(
    livestock_collection
    .filterDate("2022-01-01", "2023-01-01")
    .first()
).clip(aoi)

selected_livestock = livestock_2022.select([
    "cattle",
    "sheep",
    "goat",
    "horse"
])

total_selected_livestock = (
    selected_livestock
    .reduce(ee.Reducer.sum())
    .rename("cattle_sheep_goat_horse_2022")
)

livestock_export = selected_livestock.addBands(
    total_selected_livestock
)

livestock_csg = (
    livestock_2022
    .select(["cattle", "sheep", "goat"])
    .reduce(ee.Reducer.sum())
    .rename("cattle_sheep_goat_2022")
)

## Code block 6: Trends.Earth SDG 15.3.1 — degradation screening

### Methodological objective
This block identifies persistent rangeland pixels that also show evidence of land degradation. This intersection becomes the ecological “core” from which restoration suitability is screened.

### Line-by-line
- `ee.Image("...TrendsEarth_SDG15-3-1_2000-2023...")` — loads the Trends.Earth SDG 15.3.1 raster.
- `.select([13])` — selects band index 13 from that multi-band raster.
- `.rename("TE_status_2023")` — assigns a readable name.
- `.clip(aoi)` — clips to the AOI.

### Broad degradation mask
- `te_status_2023.gte(1)` — value must be at least 1.
- `.And(te_status_2023.lte(3))` — value must also be at most 3.
- `updateMask(...)` — keeps only values 1-3.

`restoration_need` then intersects those retained degradation-status pixels with `persistent_natsemi`.

### Core degradation criterion
`active_degradation` is true when:
- status == 1, or
- status == 2.

`core_restoration` then starts from persistent GPW rangeland and masks it using `active_degradation`.

### Result
`core_restoration` is the main ecological base mask used later:  
**persistent GPW natural/semi-natural rangeland ∩ selected Trends.Earth degradation statuses**.

### Documentation note
The code establishes which numerical status values are used, but it does not document the semantic label associated with each Trends.Earth status code. Those labels should be checked against the metadata of the exact Trends.Earth asset before writing them into a methodology report.


In [ ]:
# ================================================================
# 6. TRENDS.EARTH SDG 15.3.1
# ================================================================

trends_earth = ee.Image(
    "projects/gef-land-degradation/assets/"
    "TrendsEarth_SDG15-3-1_2000-2023_Trends-Earth"
)

te_status_2023 = (
    trends_earth
    .select([13])
    .rename("TE_status_2023")
    .clip(aoi)
)

te_degradation = te_status_2023.updateMask(
    te_status_2023.gte(1)
    .And(te_status_2023.lte(3))
)

restoration_need = (
    te_degradation
    .updateMask(persistent_natsemi)
    .rename("rangeland_restoration_need")
)

active_degradation = (
    te_status_2023.eq(1)
    .Or(te_status_2023.eq(2))
)

core_restoration = (
    persistent_natsemi
    .updateMask(active_degradation)
    .rename("core_restoration_candidate")
)

## Code block 7: CHIRPS rainfall context

### Methodological objective
This section evaluates whether recent rainy seasons were drier than a historical reference period.

The seasonal window is **15 July to 31 October**.

### Dataset
- `UCSB-CHG/CHIRPS/DAILY`
- `.select("precipitation")` — keeps daily precipitation.

### Function `seasonal_rain(year)`
- Converts `year` to integer.
- Defines `start` = 15 July.
- Defines `end` = 1 November, so observations through 31 October are included.
- `.filterDate(start, end)` — selects daily rainfall in that seasonal window.
- `.sum()` — sums rainfall through the season.
- `.rename("seasonal_rain_mm")` — names the output.
- `.set("year", year)` — adds year metadata.

### Historical rainfall baseline
`[seasonal_rain(y) for y in range(2001, 2021)]` creates one seasonal-total image for each year 2001-2020.

`.mean()` computes the **mean seasonal rainfall** across those 20 years.

### 2025 rainfall anomaly
`rain_2025.subtract(baseline_rain).divide(baseline_rain).multiply(100)` calculates:

`100 × (Rain_2025 - Mean_2001_2020) / Mean_2001_2020`

Therefore:
- negative values = drier than the baseline mean,
- positive values = wetter than the baseline mean.

### Generic annual anomaly function
`seasonal_rain_anomaly(year)` applies the same percentage anomaly formula to any selected year.

### Dry-year Boolean collection
- `drought_years = [2021, 2022, 2023, 2024, 2025]`
- `.lte(RAIN_DRY_THRESHOLD_PCT)` — marks a year as dry where the anomaly is ≤ -10%.
- `.toByte()` — converts Boolean values to 0/1.
- The list comprehension creates one 0/1 dry-season raster per year.

These annual dry masks are summed later to count how many dry seasons occurred from 2021 to 2025.


In [ ]:
# ================================================================
# 7. CHIRPS RAINFALL CONTEXT
# Jul 15 - Oct 31
# ================================================================

chirps = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .select("precipitation")
)


def seasonal_rain(year):
    year = int(year)

    start = ee.Date.fromYMD(year, 7, 15)
    end = ee.Date.fromYMD(year, 11, 1)

    return (
        chirps
        .filterDate(start, end)
        .sum()
        .rename("seasonal_rain_mm")
        .set("year", year)
    )


baseline_rain = ee.ImageCollection.fromImages(
    [seasonal_rain(y) for y in range(2001, 2021)]
).mean()

rain_2025 = seasonal_rain(2025)

rain_anomaly_2025 = (
    rain_2025
    .subtract(baseline_rain)
    .divide(baseline_rain)
    .multiply(100)
    .rename("rain_anomaly_pct_2025")
    .clip(aoi)
)


def seasonal_rain_anomaly(year):
    return (
        seasonal_rain(year)
        .subtract(baseline_rain)
        .divide(baseline_rain)
        .multiply(100)
        .rename("rain_anomaly_pct")
    )


drought_years = [2021, 2022, 2023, 2024, 2025]

annual_rain_dry = ee.ImageCollection.fromImages([
    seasonal_rain_anomaly(y)
    .lte(RAIN_DRY_THRESHOLD_PCT)
    .rename("rain_dry")
    .toByte()
    .set("year", y)
    for y in drought_years
])

## Code block 8: GloFAS 100-year river flood hazard

### Methodological objective
Flood hazard is treated as a **constraint**: areas with excessive modeled flood depth are excluded from restoration suitability.

### Line-by-line
- `ee.ImageCollection("JRC/CEMS_GLOFAS/FloodHazard/v2_1")` — loads the GloFAS flood-hazard collection.
- `.filterBounds(aoi)` — keeps images intersecting the AOI.

### RP100 depth raster
- `.select("RP100_depth")` — selects modeled water depth for the 100-year return-period event.
- `.mosaic()` — mosaics the spatial tiles into a single image.
- `.clip(aoi)` — clips to the AOI.
- `.rename("RP100_flood_depth_m")` — names the depth band.

### Flood extent
- `flood_100_depth.gte(FLOOD_MIN_DEPTH_M)` — pixels with depth ≥ 0.1 m are considered part of the modeled flood extent.
- `.selfMask()` — removes non-flood pixels from display/analysis.
- `.rename("RP100_flood_extent")` — names the binary extent layer.


In [ ]:
# ================================================================
# 8. RIVER FLOOD HAZARD - GloFAS RP100
# ================================================================

glofas_flood = (
    ee.ImageCollection("JRC/CEMS_GLOFAS/FloodHazard/v2_1")
    .filterBounds(aoi)
)

flood_100_depth = (
    glofas_flood
    .select("RP100_depth")
    .mosaic()
    .clip(aoi)
    .rename("RP100_flood_depth_m")
)

flood_100_extent = (
    flood_100_depth
    .gte(FLOOD_MIN_DEPTH_M)
    .selfMask()
    .rename("RP100_flood_extent")
)

## Code block 9: Seasonal VCI

### Methodological objective
The VCI component measures how poor the vegetation condition of a pixel is relative to that pixel's own historical seasonal range.

This implementation uses:
- seasonal window: **15 July-31 October**
- baseline: **2001-2020**
- analysis: **2021-2025**
- threshold: **VCI ≤ 0.40**

This is a seasonal VCI, not a month-matched September-only VCI.

### MODIS collection
- `ee.ImageCollection("MODIS/061/MOD13A1")` — loads MODIS vegetation indices at nominal 500 m resolution.
- `.select(VCI_VEG_BAND)` — selects `NDVI`.
- `.filterBounds(aoi)` — spatially filters the collection.
- `modis_vci_projection = ...projection()` — stores the native projection from the first image.

### Function `seasonal_mean_vi(year)`
- Defines the July 15-November 1 date interval.
- `.filterDate(start, end)` — selects MODIS observations during the rainy-season window.
- `.mean()` — computes the seasonal **mean** NDVI for that year.
- `.setDefaultProjection(modis_vci_projection)` — keeps the MODIS reference projection.
- `.rename("VI")` — generic vegetation-index name.
- `.clip(aoi)` — clips to the AOI.
- The returned image receives `year` and `system:time_start` properties.

### Historical seasonal reference
`seasonal_vi_baseline` contains one seasonal-mean NDVI image for every year 2001-2020.

- `.min()` — historical minimum seasonal mean NDVI at each pixel.
- `.max()` — historical maximum seasonal mean NDVI at each pixel.

These are **pixel-specific** minima and maxima.

### Function `calculate_seasonal_vci(year)`
For each analysis year:

1. `current_vi = seasonal_mean_vi(year)` obtains that year's seasonal mean.
2. `denominator = seasonal_vi_max - seasonal_vi_min`.
3. `denominator.where(denominator.eq(0), 0.0001)` prevents division by zero.
4. VCI is calculated as:

`VCI = (Current seasonal VI - Historical minimum) / (Historical maximum - Historical minimum)`

5. The output is named `VCI`, converted to float, clipped, and tagged with year/time metadata.

### Analysis collection
`seasonal_vci_collection` computes VCI for every year from 2021 through 2025.

### Annual vegetation-stress mask
`img.lte(VCI_DROUGHT_THRESHOLD)` produces 1 where VCI ≤ 0.40 and 0 otherwise.

### Recurrence
- `.sum()` across 2021-2025 gives `VCI_drought_count_2021_2025`, i.e. the number of seasons classified as stressed.
- `.gte(VCI_MIN_DROUGHT_YEARS)` identifies recurrent stress where at least 2 of 5 seasons are stressed.
- `core_recurrent_vci_drought` intersects recurrent VCI stress with the ecological restoration core.
- `vci_2025` extracts the 2025 VCI image for export.

### Important methodological limitations
- The code does **not** apply a MODIS QA/cloud-quality mask.
- The code uses a seasonal **mean**, not a median.
- Raw MODIS NDVI scale is not explicitly multiplied by its scale factor. For the VCI ratio, a constant multiplicative scale factor cancels mathematically, but raw values should still be understood correctly when inspected independently.
- Pixels with zero historical NDVI range are not masked; the denominator is replaced by `0.0001`. These pixels deserve QA because a very small artificial denominator can produce unstable values.
- VCI is evidence of relative vegetation stress, not by itself proof of meteorological drought.


In [ ]:
# ================================================================
# 9. SEASONAL VCI - ORIGINAL GCF LOGIC
# Jul 15 - Oct 31, analysis 2021-2025
# baseline 2001-2020
#
# NOTE:
# This reproduces the VCI logic from the original script.
# It is NOT the separate September-vs-September VCI analysis.
# ================================================================

modis_vci = (
    ee.ImageCollection("MODIS/061/MOD13A1")
    .select(VCI_VEG_BAND)
    .filterBounds(aoi)
)

modis_vci_projection = ee.Image(
    modis_vci.first()
).projection()


def seasonal_mean_vi(year):
    year = int(year)

    start = ee.Date.fromYMD(year, 7, 15)
    end = ee.Date.fromYMD(year, 11, 1)

    seasonal_mean = (
        modis_vci
        .filterDate(start, end)
        .mean()
        .setDefaultProjection(modis_vci_projection)
        .rename("VI")
        .clip(aoi)
    )

    return seasonal_mean.set({
        "year": year,
        "system:time_start": start.millis()
    })


seasonal_vi_baseline = ee.ImageCollection.fromImages([
    seasonal_mean_vi(y)
    for y in range(
        VCI_BASELINE_START_YEAR,
        VCI_BASELINE_END_YEAR + 1
    )
])

seasonal_vi_min = (
    seasonal_vi_baseline
    .min()
    .setDefaultProjection(modis_vci_projection)
)

seasonal_vi_max = (
    seasonal_vi_baseline
    .max()
    .setDefaultProjection(modis_vci_projection)
)


def calculate_seasonal_vci(year):
    year = int(year)

    current_vi = (
        seasonal_mean_vi(year)
        .setDefaultProjection(modis_vci_projection)
    )

    denominator = (
        seasonal_vi_max
        .subtract(seasonal_vi_min)
        .setDefaultProjection(modis_vci_projection)
    )

    safe_denominator = denominator.where(
        denominator.eq(0),
        0.0001
    )

    vci = (
        current_vi
        .subtract(seasonal_vi_min)
        .divide(safe_denominator)
        .rename("VCI")
        .toFloat()
        .clip(aoi)
    )

    return vci.set({
        "year": year,
        "system:time_start":
            ee.Date.fromYMD(year, 7, 15).millis()
    })


seasonal_vci_collection = ee.ImageCollection.fromImages([
    calculate_seasonal_vci(y)
    for y in range(
        VCI_ANALYSIS_START_YEAR,
        VCI_ANALYSIS_END_YEAR + 1
    )
])

annual_vci_drought = seasonal_vci_collection.map(
    lambda img:
        img.lte(VCI_DROUGHT_THRESHOLD)
        .rename("VCI_drought")
        .toByte()
        .copyProperties(
            img,
            ["year", "system:time_start"]
        )
)

vci_drought_count = (
    annual_vci_drought
    .sum()
    .rename("VCI_drought_count_2021_2025")
    .clip(aoi)
)

recurrent_vci_drought = (
    vci_drought_count
    .gte(VCI_MIN_DROUGHT_YEARS)
    .selfMask()
    .rename("recurrent_VCI_drought")
)

core_recurrent_vci_drought = (
    core_restoration
    .updateMask(recurrent_vci_drought)
    .rename("core_candidate_recurrent_VCI_drought")
)

vci_2025 = ee.Image(
    seasonal_vci_collection
    .filter(ee.Filter.eq("year", 2025))
    .first()
)

## Code block 10: target provinces

### Methodological objective
The broad AOI is narrowed to the provinces used for the final restoration suitability analysis.

### Line-by-line
- `target_province_codes = [...]` defines:
  - `TCD14` — Ouaddai
  - `TCD21` — Sila
  - `TCD15` — Salamat
- `provinces.filter(ee.Filter.inList(...))` — selects features whose `ADM1_PCODE` is one of those three codes.
- `target_aoi = target_provinces.geometry()` — converts the selected provinces to the final target geometry.

### Number of dry seasons
- `annual_rain_dry.sum()` — sums the 0/1 dry-season masks for 2021-2025.
- The resulting pixel value ranges from 0 to 5 and represents the number of recent seasons with rainfall anomaly ≤ -10%.
- `.clip(target_aoi)` restricts this count to the three target provinces.


In [ ]:
# ================================================================
# 10. TARGET PROVINCES
# Ouaddai + Sila + Salamat
# ================================================================

target_province_codes = [
    "TCD14",  # Ouaddai
    "TCD21",  # Sila
    "TCD15"   # Salamat
]

target_provinces = provinces.filter(
    ee.Filter.inList(
        "ADM1_PCODE",
        target_province_codes
    )
)

target_aoi = target_provinces.geometry()

rain_dry_count = (
    annual_rain_dry
    .sum()
    .rename("rain_dry_year_count_2021_2025")
    .clip(target_aoi)
)

## Code block 11: livestock thresholds

### Methodological objective
This section derives **relative livestock thresholds** from the data rather than imposing arbitrary fixed head-count values.

The thresholds are calculated only in 1 km livestock cells that overlap the ecological restoration core.

### Projections
- `livestock_projection = livestock_csg.projection()` — stores the 1 km livestock grid projection.
- `gpw_projection = ...annual_natsemi_collection.first().projection()` — stores the GPW rangeland projection.

### Preparing the restoration core
- `core_restoration.unmask(0)` — makes non-core pixels explicit zeros.
- `.setDefaultProjection(gpw_projection)` — uses the GPW projection as the fine-grid reference.

### Aggregating core coverage to the livestock grid
- `.reduceResolution(reducer=ee.Reducer.mean(), maxPixels=4096)` — aggregates the fine binary core mask to coarser livestock cells using the mean.
- Because the input is 0/1, the mean is the approximate **fraction of each 1 km livestock cell covered by the restoration core**.
- `.reproject(livestock_projection)` — aligns the result to the livestock grid.
- `.rename("core_fraction")` — names the fractional overlap.

### Masking livestock to relevant cells
- `core_fraction_livestock_grid.gt(0)` — keeps any livestock cell with at least some restoration-core overlap.
- `livestock_csg.clip(target_aoi)` — limits livestock to the target provinces.
- `.updateMask(core_livestock_mask)` — retains only cells overlapping the restoration core.
- `.updateMask(livestock_csg.gt(0))` — excludes zero-livestock cells.
- `.rename("livestock")` — simplifies the band name.

### Percentile thresholds
`reduceRegion(reducer=ee.Reducer.percentile([50, 75]), ...)` calculates:
- the 50th percentile (median) livestock value,
- the 75th percentile livestock value,

across the relevant target-area cells at 1 km scale.

- `livestock_p50` retrieves the median.
- `livestock_p75` retrieves the 75th percentile.

### Resulting evidence masks
- `moderate_livestock = livestock_csg.gte(livestock_p50)` — moderate evidence where livestock ≥ median.
- `conservative_livestock = livestock_csg.gte(livestock_p75)` — stricter evidence where livestock ≥ 75th percentile.

This makes the livestock criterion relative to the distribution of livestock density/headcount in candidate rangeland areas.


In [ ]:
# ================================================================
# 11. LIVESTOCK THRESHOLDS
# ================================================================

livestock_projection = livestock_csg.projection()

gpw_projection = ee.Image(
    annual_natsemi_collection.first()
).projection()

core_for_livestock_aggregation = (
    core_restoration
    .unmask(0)
    .setDefaultProjection(gpw_projection)
)

core_fraction_livestock_grid = (
    core_for_livestock_aggregation
    .reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=4096
    )
    .reproject(livestock_projection)
    .rename("core_fraction")
)

core_livestock_mask = core_fraction_livestock_grid.gt(0)

livestock_for_threshold = (
    livestock_csg
    .clip(target_aoi)
    .updateMask(core_livestock_mask)
    .updateMask(livestock_csg.gt(0))
    .rename("livestock")
)

livestock_thresholds = livestock_for_threshold.reduceRegion(
    reducer=ee.Reducer.percentile([50, 75]),
    geometry=target_aoi,
    scale=1000,
    maxPixels=1e13,
    tileScale=4
)

livestock_p50 = ee.Number(
    livestock_thresholds.get("livestock_p50")
)

livestock_p75 = ee.Number(
    livestock_thresholds.get("livestock_p75")
)

moderate_livestock = livestock_csg.gte(livestock_p50)
conservative_livestock = livestock_csg.gte(livestock_p75)

## Code block 12: flood constraints

### Methodological objective
This converts continuous RP100 flood depth into two suitability masks.

### Line-by-line
- `flood_100_depth.unmask(0)` — replaces masked/no-data flood-depth pixels with zero.
- `.clip(target_aoi)` — restricts flood depth to the target provinces.

### Moderate flood rule
`flood_depth_filled.lt(MODERATE_MAX_FLOOD_DEPTH_M)`  
With the current parameter, a pixel passes if modeled RP100 depth is **< 0.5 m**.

### Conservative flood rule
`flood_depth_filled.lt(FLOOD_MIN_DEPTH_M)`  
With the current parameter, a pixel passes if modeled RP100 depth is **< 0.1 m**.

### Important assumption
Because `unmask(0)` is applied first, missing GloFAS values are treated as zero flood depth and therefore pass both flood filters. This should be stated explicitly in the methodology and, ideally, checked spatially.


In [ ]:
# ================================================================
# 12. FLOOD CONSTRAINTS
# ================================================================

flood_depth_filled = (
    flood_100_depth
    .unmask(0)
    .clip(target_aoi)
)

moderate_flood_ok = flood_depth_filled.lt(
    MODERATE_MAX_FLOOD_DEPTH_M
)

conservative_flood_ok = flood_depth_filled.lt(
    FLOOD_MIN_DEPTH_M
)

## Code block 13: final restoration suitability

### Methodological objective
This is the decision engine of the workflow. It combines the ecological core with drought/stress, livestock and flood evidence.

There are two levels:

1. **Moderate suitability** — flexible evidence rule.
2. **Conservative suitability** — all key conditions must be satisfied.

### VCI evidence
- `moderate_vci = vci_drought_count.gte(1)` — at least one stressed VCI season during 2021-2025.
- `conservative_vci = vci_drought_count.gte(2)` — at least two stressed VCI seasons.

### Rainfall evidence
- `rainfall_evidence = rain_dry_count.gte(1)` — at least one season during 2021-2025 had rainfall ≤ -10% relative to the 2001-2020 seasonal mean.

### Moderate evidence count
Three Boolean indicators are converted to bytes and added:
1. moderate VCI evidence,
2. rainfall evidence,
3. moderate livestock evidence.

`moderate_evidence_count` therefore ranges from 0 to 3.

### Moderate suitability
A pixel is retained only if:
- it is already in `core_restoration`,
- at least **2 of the 3** evidence variables above are true,
- RP100 flood depth is **< 0.5 m**.

This is an **OR-like multi-evidence rule**: not every evidence layer must pass.

### Conservative suitability
A pixel is retained only if **all** of the following are true:
- it is in `core_restoration`,
- VCI stress occurred in at least 2 analysis years,
- at least one dry rainfall season occurred,
- cattle + sheep + goat abundance is at or above the 75th percentile,
- RP100 flood depth is < 0.1 m.

This is an **AND rule**, so it is substantially stricter.

### Removing overlap
`moderate_only_suitable` starts from the moderate mask and removes pixels also classified as conservative.

### Final categorical raster
`ee.Image(0)` creates a base image.

- `.where(moderate_only_suitable.mask(), 1)` assigns class 1 to moderate-only pixels.
- `.where(conservative_suitable.mask(), 2)` assigns class 2 to conservative pixels.
- `.updateMask(...)` removes all locations belonging to neither class.
- `.rename("restoration_suitability_level")` gives the final band a descriptive name.
- `.toByte()` stores the result as an 8-bit integer raster.

### Final class meaning
- `1` = moderate-only restoration suitability
- `2` = conservative restoration suitability
- masked = not selected by either screening rule


In [ ]:
# ================================================================
# 13. RESTORATION SUITABILITY
# ================================================================

moderate_vci = vci_drought_count.gte(
    MODERATE_MIN_VCI_DROUGHT_YEARS
)

conservative_vci = vci_drought_count.gte(
    CONSERVATIVE_MIN_VCI_DROUGHT_YEARS
)

rainfall_evidence = rain_dry_count.gte(1)

moderate_evidence_count = (
    moderate_vci.toByte()
    .add(rainfall_evidence.toByte())
    .add(moderate_livestock.toByte())
    .rename("moderate_evidence_count")
)

moderate_suitable = (
    core_restoration
    .clip(target_aoi)
    .updateMask(moderate_evidence_count.gte(2))
    .updateMask(moderate_flood_ok)
    .rename("moderate_restoration_suitability")
)

conservative_suitable = (
    core_restoration
    .clip(target_aoi)
    .updateMask(conservative_vci)
    .updateMask(rainfall_evidence)
    .updateMask(conservative_livestock)
    .updateMask(conservative_flood_ok)
    .rename("conservative_restoration_suitability")
)

moderate_only_suitable = (
    moderate_suitable
    .updateMask(
        conservative_suitable.mask().eq(0)
    )
    .rename("moderate_only_restoration_suitability")
)

suitability_level = (
    ee.Image(0)
    .clip(target_aoi)
    .where(moderate_only_suitable.mask(), 1)
    .where(conservative_suitable.mask(), 2)
    .updateMask(
        moderate_only_suitable.mask()
        .Or(conservative_suitable.mask())
    )
    .rename("restoration_suitability_level")
    .toByte()
)

## Code block 14: local download helper

### Methodological role
This block does not change the analytical results. It standardizes how Earth Engine images are exported to local GeoTIFF files.

### Function arguments
- `image` — Earth Engine image to download.
- `filename` — local output filename.
- `region` — geometry used as the export extent.
- `scale` — requested pixel size in metres.
- `crs="EPSG:4326"` — output CRS defaults to geographic WGS 84.

### Line-by-line behavior
- `out_file = OUTPUT_DIR / filename` — constructs the full output path.
- The `print(...)` statements show progress, resolution and destination.
- `try:` begins protected execution so one failed download does not crash silently.
- `geemap.download_ee_image(...)` performs the local download.
- `image=image` passes the Earth Engine raster.
- `filename=str(out_file)` supplies the full output path.
- `region=region` clips the requested export extent.
- `scale=scale` requests the nominal pixel size.
- `crs=crs` controls output CRS.
- `overwrite=True` allows an existing file with the same name to be replaced.
- `num_threads=4` is passed to geemap/geedim; the execution log notes that this argument is deprecated in the installed version.
- `max_tile_dim=2048` allows large downloads to be split into manageable tiles.
- `out_file.exists()` checks whether the file actually appeared.
- File size is converted from bytes to MB for reporting.
- `except Exception as exc:` prints the error if a download fails and recommends using an Earth Engine batch export for particularly heavy products.

### GIS note
The default output CRS is `EPSG:4326`. Therefore, although `scale` is requested in metres, the stored GeoTIFF coordinate system is geographic. For later **area calculations in ArcGIS Pro**, reproject the raster/analysis to an appropriate projected CRS before interpreting polygon/raster areas in square metres or hectares.


In [ ]:
# ================================================================
# 14. LOCAL DOWNLOAD HELPER
# ================================================================

def download_raster(
    image,
    filename,
    region,
    scale,
    crs="EPSG:4326"
):
    """
    Download an Earth Engine image directly to the local PC.

    geemap.download_ee_image uses geedim internally and can tile
    large downloads. Derived products over very large AOIs can
    nevertheless remain computationally heavy.
    """

    out_file = OUTPUT_DIR / filename

    print("\n" + "=" * 70)
    print(f"Downloading: {filename}")
    print(f"Scale: {scale} m")
    print(f"Destination: {out_file}")

    try:
        geemap.download_ee_image(
            image=image,
            filename=str(out_file),
            region=region,
            scale=scale,
            crs=crs,
            overwrite=True,
            num_threads=4,
            max_tile_dim=2048
        )

        if out_file.exists():
            size_mb = out_file.stat().st_size / (1024 ** 2)
            print(f"OK: {filename} ({size_mb:.1f} MB)")
        else:
            print(
                "WARNING: geemap finished but the file "
                "was not found at the expected path."
            )

    except Exception as exc:
        print(f"FAILED: {filename}")
        print(exc)
        print(
            "\nIf this specific derived raster is too computationally "
            "large for direct download, set its DOWNLOAD flag to False "
            "and export that one through an Earth Engine batch task."
        )

## Code block 15: download rasters

### Methodological role
This block simply exports the intermediate and final products created above. Each output is controlled by a `DOWNLOAD[...]` switch.

### Repeated pattern
Every block follows:

`if DOWNLOAD["name"]:`  
→ check whether that product should be exported.

`download_raster(...)`  
→ send the Earth Engine image to the helper function.

Inside each call:
1. the first argument is the image,
2. the second is the filename,
3. the third is the export geometry (`aoi` or `target_aoi`),
4. the fourth is the requested output scale.

### Outputs and requested scales

- `TCD_GPW_natsemi_persistence_2018_2022.tif` — 30 m; number of qualifying GPW natural/semi-natural years.
- `TCD_GPW_persistent_natsemi_2018_2022.tif` — 30 m; persistent GPW rangeland mask.
- `TCD_GPW_mean_natsemi_probability_2018_2022.tif` — 30 m; mean GPW natural/semi-natural probability.
- `TCD_GPW_WC_cropland_conflict.tif` — 30 m; diagnostic GPW-vs-WorldCover cropland conflict, only if its switch is enabled.
- `TCD_GPW_livestock_2022_cattle_sheep_goat_horse.tif` — 1 km; livestock bands and total.
- `TCD_GPW_TrendsEarth_rangeland_restoration_need.tif` — 250 m export; degraded persistent rangeland screening.
- `TCD_core_restoration_candidate.tif` — 30 m export; persistent rangeland intersecting selected degradation status.
- `TCD_CHIRPS_JulOct_rain_anomaly_2025.tif` — 5.5 km export; 2025 seasonal rainfall anomaly.
- `TCD_GloFAS_RP100_flood_depth_m.tif` — 90 m export; RP100 flood depth.
- `TCD_GloFAS_RP100_flood_extent.tif` — 90 m export; binary RP100 flood extent.
- `TCD_seasonal_VCI_JulOct_2025.tif` — 500 m export; seasonal VCI for 2025.
- `TCD_VCI_drought_count_2021_2025.tif` — 500 m export; number of VCI-stress seasons.
- `TCD_core_candidate_recurrent_VCI_drought.tif` — 500 m export; ecological core intersecting recurrent VCI stress.
- `TCD_restoration_suitability_moderate_conservative.tif` — 30 m export; final two-class suitability raster.
- `TCD_restoration_suitability_conservative.tif` — 30 m export; conservative mask only.

The final `print(...)` statements mark completion and remind the user where the files were saved.

### Resolution warning
Requested export resolution and **effective information resolution are not the same thing**.  
For example, the final suitability raster is exported at 30 m but combines information from coarser inputs such as 500 m VCI, ~5.5 km CHIRPS rainfall, 1 km livestock and 90 m flood hazard. The output should therefore not be described as if all suitability evidence were independently resolved at 30 m.


In [ ]:
# ================================================================
# 15. DOWNLOAD RASTERS
# ================================================================

if DOWNLOAD["gpw_persistence_count"]:
    download_raster(
        persistence_count.toByte(),
        f"TCD_GPW_natsemi_persistence_{START_YEAR}_{END_YEAR}.tif",
        aoi,
        30
    )

if DOWNLOAD["gpw_persistent_rangeland"]:
    download_raster(
        persistent_natsemi.toByte(),
        f"TCD_GPW_persistent_natsemi_{START_YEAR}_{END_YEAR}.tif",
        aoi,
        30
    )

if DOWNLOAD["gpw_mean_probability"]:
    download_raster(
        mean_natsemi_prob.toFloat(),
        f"TCD_GPW_mean_natsemi_probability_{START_YEAR}_{END_YEAR}.tif",
        aoi,
        30
    )


if DOWNLOAD["gpw_worldcover_crop_conflict"]:
    download_raster(
        gpw_cropland_conflict.toByte(),
        "TCD_GPW_WC_cropland_conflict.tif",
        aoi,
        30
    )

if DOWNLOAD["livestock_2022"]:
    download_raster(
        livestock_export.toFloat(),
        "TCD_GPW_livestock_2022_cattle_sheep_goat_horse.tif",
        aoi,
        1000
    )

if DOWNLOAD["trends_earth_restoration_need"]:
    download_raster(
        restoration_need.toByte(),
        "TCD_GPW_TrendsEarth_rangeland_restoration_need.tif",
        aoi,
        250
    )

if DOWNLOAD["core_restoration_candidate"]:
    download_raster(
        core_restoration.toByte(),
        "TCD_core_restoration_candidate.tif",
        target_aoi,
        30
    )

if DOWNLOAD["chirps_rain_anomaly_2025"]:
    download_raster(
        rain_anomaly_2025.toFloat(),
        "TCD_CHIRPS_JulOct_rain_anomaly_2025.tif",
        aoi,
        5500
    )

if DOWNLOAD["glofas_rp100_depth"]:
    download_raster(
        flood_100_depth.toFloat(),
        "TCD_GloFAS_RP100_flood_depth_m.tif",
        target_aoi,
        90
    )

if DOWNLOAD["glofas_rp100_extent"]:
    download_raster(
        flood_100_extent.toByte(),
        "TCD_GloFAS_RP100_flood_extent.tif",
        target_aoi,
        90
    )

if DOWNLOAD["vci_2025"]:
    download_raster(
        vci_2025.toFloat(),
        "TCD_seasonal_VCI_JulOct_2025.tif",
        target_aoi,
        500
    )

if DOWNLOAD["vci_drought_count_2021_2025"]:
    download_raster(
        vci_drought_count.toByte(),
        "TCD_VCI_drought_count_2021_2025.tif",
        target_aoi,
        500
    )

if DOWNLOAD["core_recurrent_vci_drought"]:
    download_raster(
        core_recurrent_vci_drought.toByte(),
        "TCD_core_candidate_recurrent_VCI_drought.tif",
        target_aoi,
        500
    )

if DOWNLOAD["suitability_moderate_conservative"]:
    download_raster(
        suitability_level,
        "TCD_restoration_suitability_moderate_conservative.tif",
        target_aoi,
        30
    )

if DOWNLOAD["suitability_conservative"]:
    download_raster(
        conservative_suitable.toByte(),
        "TCD_restoration_suitability_conservative.tif",
        target_aoi,
        30
    )


print("\n" + "=" * 70)
print("DOWNLOADS FINISHED")
print(f"Check folder:\n{OUTPUT_DIR}")

# Methodological synthesis of the final screening

The final workflow can be summarized as follows:

### Step 1 — Define persistent rangeland
A pixel is considered persistent natural/semi-natural rangeland when GPW:
- classifies it as the selected natural/semi-natural class,
- gives it probability ≥ 60%,
- and this occurs in at least 4 of the 5 years from 2018 to 2022.

### Step 2 — Define the restoration core
Persistent rangeland is intersected with selected Trends.Earth degradation-status values. This creates `core_restoration`.

### Step 3 — Build recent stress / use evidence
Three main supporting variables are created:
- VCI stress recurrence, 2021-2025;
- CHIRPS seasonal rainfall deficit, 2021-2025;
- relative livestock concentration based on the 50th and 75th percentiles.

### Step 4 — Apply flood constraints
GloFAS RP100 flood depth is used to avoid areas exceeding:
- 0.5 m for moderate screening,
- 0.1 m for conservative screening.

### Step 5 — Produce suitability classes
**Moderate:** restoration core + at least 2 of 3 evidence criteria + flood depth < 0.5 m.

**Conservative:** restoration core + recurrent VCI stress + rainfall-deficit evidence + high livestock + flood depth < 0.1 m.

The final raster contains:
- `1 = moderate-only`
- `2 = conservative`

## Main limitations to report
1. This is a **screening model**, not a causal or predictive restoration model.
2. Input datasets have different spatial resolutions; the 30 m output grid does not imply 30 m certainty.
3. The VCI implementation does not include explicit MODIS QA filtering.
4. The VCI denominator safeguard replaces zero historical range with 0.0001 rather than masking such pixels.
5. Missing GloFAS flood depth is treated as zero.
6. Livestock thresholds are relative percentiles, so their meaning depends on the selected AOI and candidate-mask distribution.
7. WorldCover is used as a diagnostic consistency layer rather than as a formal exclusion criterion.
8. The exact semantic meaning of Trends.Earth numerical classes selected from band 13 should be verified against the metadata of the source asset before publication.
